In [151]:
import sqlite3
import pandas as pd
import geopandas as gpd
pd.options.mode.chained_assignment = None  # default='warn'  --> catches unnecessary pandas warning
#import googlemaps   
from datetime import datetime
#from auth import auth

import io
from PIL import Image

In [152]:
conn = sqlite3.connect("../data/processed/plz.sqlite")
cur = conn.cursor()

In [153]:
plz_areas = gpd.read_file("../data/external/plz-5stellig.shp/plz-5stellig.shp")
# plz_areas = gpd.read_file("plz-5stellig.geojson")

In [154]:
# plz_areas["dist_car"] = 0
# plz_areas["time_car"] = 0
# plz_areas["dist_trans"] = 0
# plz_areas["time_trans"] = 0
# plz_areas["retrieved"] = 0

In [155]:
plz_areas.head()

,plz,note,einwohner,qkm,geometry
0,64743,"Situation unklar, evtl. haben die HÃ¤user Marb...",3,0.082066,"POLYGON ((8.98124 49.60761, 8.98312 49.60748, ..."
1,81248,81248 MÃ¼nchen,121,1.984763,"POLYGON ((11.39468 48.14729, 11.39490 48.14780..."
2,60315,60315 Frankfurt am Main (FOUR),0,0.017285,"POLYGON ((8.67254 50.11264, 8.67258 50.11265, ..."
3,99331,99331 Geratal,4523,20.207080,"POLYGON ((10.79153 50.69477, 10.79178 50.69819..."
4,60312,60312 Frankfurt am Main (Omniturm),0,0.001829,"POLYGON ((8.67262 50.11164, 8.67311 50.11182, ..."


In [156]:
if len(plz_areas) - len (plz_areas.drop_duplicates(subset=["plz"])) != 0:
    print("PLZ duplicates?")
else:
    print("ok")

ok


In [157]:
cur.execute("SELECT * FROM Plz WHERE google_retrieved = 1")
db_travel = pd.DataFrame(cur.fetchall(), columns=["id" ,   "plz","lat" ,"lng" ,                
                "car_distance_lmu" ,"car_time_lmu" , 
                "car_distance_tum" ,"car_time_tum" ,
                "car_distance_würzburg" ,  "car_time_würzburg" ,
                "car_distance_erlangen" , "car_time_erlangen" ,
                "car_distance_regensburg" , "car_time_regensburg" ,
                "car_distance_augsburg" , "car_time_augsburg" ,
                "transit_distance_lmu" ,  "transit_time_lmu" ,
                "transit_distance_tum" ,  "transit_time_tum" ,
                "transit_distance_würzburg" , "transit_time_würzburg" ,
                "transit_distance_erlangen" , "transit_time_erlangen" ,
                "transit_distance_regensburg" , "transit_time_regensburg" ,
                "transit_distance_augsburg" ,  "transit_time_augsburg" ,
                "google_retrieved" ,
                "date_calc" ])


In [158]:
db_travel.head()

,id,plz,lat,lng,car_distance_lmu,car_time_lmu,car_distance_tum,car_time_tum,car_distance_würzburg,car_time_würzburg,...,transit_distance_würzburg,transit_time_würzburg,transit_distance_erlangen,transit_time_erlangen,transit_distance_regensburg,transit_time_regensburg,transit_distance_augsburg,transit_time_augsburg,google_retrieved,date_calc
0,1,01067,51.057550,13.717065,475989.0,17464.0,462816.0,17208.0,365514.0,13634.0,...,508023.0,17216.0,420088.0,16708.0,411257.0,19744.0,676204.0,25140.0,1,"2025/03/13, 07:00"
1,415,07318,50.635191,11.359475,360971.0,14475.0,347797.0,14219.0,197576.0,8077.0,...,224132.0,13680.0,162394.0,10473.0,325409.0,16268.0,402557.0,18603.0,1,"2025/03/13, 07:00"
2,416,07330,50.529766,11.382800,341723.0,13841.0,328550.0,13584.0,189066.0,8415.0,...,196898.0,10744.0,136048.0,7416.0,262586.0,16332.0,296160.0,16763.0,1,"2025/03/13, 07:00"
3,417,07333,50.654978,11.438107,371412.0,14289.0,358238.0,14033.0,201157.0,8009.0,...,230241.0,14301.0,168503.0,11094.0,331518.0,16889.0,409226.0,19000.0,1,"2025/03/13, 07:00"
4,418,07334,50.630152,11.463449,369637.0,14310.0,356463.0,14053.0,205681.0,8394.0,...,235286.0,15857.0,174435.0,12469.0,336458.0,18614.0,334547.0,21576.0,1,"2025/03/13, 07:00"


In [159]:
if len(db_travel) - len (db_travel.drop_duplicates(subset=["plz"])) != 0:
    print("PLZ duplicates?")
else:
    print("ok")

ok


In [160]:
len_before = len(plz_areas)
plz_areas = pd.merge(plz_areas, db_travel, on="plz", how="left")

In [161]:
plz_areas = plz_areas.loc[plz_areas["google_retrieved"]==1]

In [162]:
for t in ["car_time_lmu", "car_time_tum", "car_time_würzburg", "car_time_erlangen", "car_time_regensburg", "car_time_augsburg",
          "transit_time_lmu", "transit_time_tum", "transit_time_würzburg", "transit_time_erlangen", "transit_time_regensburg", "transit_time_augsburg"]:
    plz_areas[t] = round((plz_areas[t] / 3600),1)

for d in ["car_distance_lmu", "car_distance_tum", "car_distance_würzburg", "car_distance_erlangen", "car_distance_regensburg", "car_distance_augsburg",
          "transit_distance_lmu", "transit_distance_tum", "transit_distance_würzburg", "transit_distance_erlangen", "transit_distance_regensburg", "transit_distance_augsburg"]:
    plz_areas[d] = round((plz_areas[d] / 3600),1)

In [163]:
plz_areas.head()

,plz,note,einwohner,qkm,geometry,id,lat,lng,car_distance_lmu,car_time_lmu,...,transit_distance_würzburg,transit_time_würzburg,transit_distance_erlangen,transit_time_erlangen,transit_distance_regensburg,transit_time_regensburg,transit_distance_augsburg,transit_time_augsburg,google_retrieved,date_calc
0,64743,"Situation unklar, evtl. haben die HÃ¤user Marb...",3,0.082066,"POLYGON ((8.98124 49.60761, 8.98312 49.60748, ...",4510.0,49.567621,8.973797,95.5,3.8,...,48.0,3.3,82.9,4.6,163.3,6.7,108.6,4.6,1.0,"2025/03/13, 07:00"
12,99707,99707 KyffhÃ¤userland,4281,128.579234,"POLYGON ((10.90105 51.32919, 10.90118 51.32948...",8258.0,51.360030,11.041520,132.5,5.0,...,81.2,4.5,64.3,3.7,109.4,5.7,144.2,6.4,1.0,"2025/03/13, 07:00"
17,60306,"60306 Frankfurt am Main, Opernturm",0,0.005761,"POLYGON ((8.66955 50.11584, 8.66990 50.11655, ...",4280.0,50.116048,8.669717,116.4,4.4,...,35.1,1.6,70.0,2.8,92.1,4.0,118.0,3.7,1.0,"2025/03/13, 07:00"
37,99819,"99819 Marksuhl, Krauthausen u.a.",1609,18.342483,"POLYGON ((10.19277 51.01861, 10.19327 51.01905...",8270.0,50.913437,10.202818,116.3,4.5,...,64.9,2.7,81.8,3.4,103.9,4.6,131.7,5.2,1.0,"2025/03/13, 07:00"
531,63930,63930 Neunkirchen,1552,16.628516,"POLYGON ((9.35572 49.70227, 9.35573 49.70231, ...",4455.0,49.704589,9.391442,98.6,3.7,...,26.0,2.9,72.8,4.6,94.9,6.5,97.6,6.2,1.0,"2025/03/13, 07:00"


In [164]:
states = gpd.read_file("../data/external/DEU_adm/DEU_adm1.shp")
bavaria = states.loc[states["NAME_1"] == "Bayern"]
bavaria

,ID_0,ISO,NAME_0,ID_1,NAME_1,TYPE_1,ENGTYPE_1,NL_NAME_1,VARNAME_1,geometry
1,86,DEU,Germany,2,Bayern,Land,State,None,Bavaria,"POLYGON ((10.13386 50.55000, 10.13980 50.54252..."


In [165]:
# plz_map_rgb = bavaria.explore(tooltip=False, highlight=False, style_kwds={"color": "#434343", "fill": False})

In [166]:
# plz_map_rgb = plz_areas.explore(                    
#                     column="car_time_erlangen",
#                     cmap="OrRd",
#                     tooltip=["plz", "car_time_erlangen", "car_distance_erlangen"],
#                     style_kwds={"stroke": True, "stroke-width": 2, "stroke-color": "black"} 
# )
# plz_map_rgb = bavaria.explore(m=plz_map_rgb, tooltip=False, highlight=False, style_kwds={"color": "#434343", "fill": False})

# plz_map_rgb

In [167]:
# plz_erlangen_transit = plz_areas.dropna(subset="transit_time_erlangen")

# plz_map_rgb = plz_erlangen_transit.explore(                    
#                     column="transit_time_erlangen",
#                     cmap="viridis_r",
#                     #color = "blue",
#                     tooltip=["plz", "transit_time_erlangen", "transit_distance_erlangen"],
#                     vmin = 0,
#                     vmax = 4,
#                     style_kwds={"stroke": True, "color": None, "weight": 0.5}#, "stroke-width": 0.1, "stroke-color": "black"} 
# )
# plz_map_rgb = bavaria.explore(m=plz_map_rgb, tooltip=False, highlight=False, style_kwds={"color": "#434343", "fill": False})

# plz_map_rgb.save("erlangen.html")

# plz_map_rgb


In [168]:
img_data = plz_map_rgb._to_png(5)
img = Image.open(io.BytesIO(img_data))
img.save('image_test_transit.png')

In [169]:
plz_areas["min_time_car"] = plz_areas[[     "car_time_lmu", 
                                            "car_time_tum",
                                            "car_time_würzburg",
                                            "car_time_erlangen",
                                            "car_time_regensburg",
                                            "car_time_augsburg"]].min(axis=1)

In [170]:
len(plz_areas)

3399

In [171]:
clinic_list = [["LMU", "car_time_lmu", "#003f5c"],
               ["TUM", "car_time_tum", "#dd5182"],
               ["Würzburg", "car_time_würzburg", "#955196"],
               ["Erlangen", "car_time_erlangen", "#444e86"],
               ["Regensburg", "car_time_regensburg", "#ff6e54"],
               ["Augsburg", "car_time_augsburg", "#ffa600"]]

In [172]:
# Generates a pandas DataFrame containing the locations of al clinica, transforms it to a GeoPandas GeoDataFrame and maps (with marker symbols) it on the two existing maps 
clinica_map = pd.DataFrame({
                        "id":   ["TUM",     "LMU",     "Würzburg", "Erlangen", "Regensburg", "Augsburg"],
                        "lat":  [48.135833, 48.111388, 49.800833,  49.599444,  48.987778,    48.385833], #Breitengrad Latitude y
                        "lon":  [11.599167, 11.469444,  9.953611,  11.010556,  12.090278,    10.837500]  #Längengrad Longitude x
                        
})

clinica_map_geo = gpd.GeoDataFrame(clinica_map,
                                   geometry=gpd.points_from_xy(clinica_map.lon, clinica_map.lat),
                                   crs="EPSG:4326"
                                   )

plz_close_map = clinica_map_geo.explore(marker_type="marker")

In [173]:
# for clinic in clinic_list:
#     plz_close = plz_areas.loc[plz_areas[clinic[1]] == plz_areas["min_time_car"]]

#     plz_close_map = plz_close.explore(     
#                     m=plz_close_map,               
#                     color = clinic[2],
#                     tooltip=["plz", "min_time_car"],                    
#                     style_kwds={"stroke": True, "color": None, "weight": 0.5}#, "stroke-width": 0.1, "stroke-color": "black"} 
#     )

# plz_close_map = bavaria.explore(m=plz_close_map, tooltip=False, highlight=False, style_kwds={"color": "#434343", "fill": False})

# plz_close_map

In [174]:
cur.close()
conn.close()